## 1. 라이브러리 및 환경 설정

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

## 2. 데이터 로드

In [3]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 94)
테스트 데이터 크기: (50000, 93)


## 3. 피처 및 타겟 설정

In [4]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


## 4. 모델 학습 (5-Fold CV)

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=7,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 494.261
[200]	valid_0's l2: 474.567
[300]	valid_0's l2: 460.88
[400]	valid_0's l2: 450.216
[500]	valid_0's l2: 442.736
[600]	valid_0's l2: 436.378
[700]	valid_0's l2: 430.24
[800]	valid_0's l2: 425.243
[900]	valid_0's l2: 421.239
[1000]	valid_0's l2: 417.524
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 417.524
── Fold 2 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 503.98
[200]	valid_0's l2: 485.04
[300]	valid_0's l2: 473.31
[400]	valid_0's l2: 464.703
[500]	valid_0's l2: 458.153
[600]	valid_0's l2: 451.458
[700]	valid_0's l2: 445.839
[800]	valid_0's l2: 440.982
[900]	valid_0's l2: 436.954
[1000]	valid_0's l2: 433.272
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 433.272
── Fold 3 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 468.227
[200]	valid_0's l2: 448.275
[300]	valid

## 5. 결과 확인

In [6]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 9.2405


## 6. 제출 파일 생성

In [7]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
